In [2]:
import sys
print(sys.executable)

/Users/raymondfurtado/Desktop/ML project End to End/venv/bin/python


In [5]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [6]:
df = pd.read_csv('stud.csv')

In [7]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [8]:
df = df.rename(columns = {'race_ethnicity': 'race/ethnicity', 'parental_level_of_education': 'parental level of education', 'test_preparation_course': 'test preparation course','lunch':'lunch type'})

In [9]:
df.head()

,gender,race/ethnicity,parental level of education,lunch type,test preparation course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [11]:
#  Preparing X and Y variables 
# The goal here is to predict the math score of the students. So, we will set the math score as our target variable and the rest of the columns as our features.

X = df.drop('math_score',axis=1)
y = df['math_score']

In [12]:
X.head()

,gender,race/ethnicity,parental level of education,lunch type,test preparation course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [15]:
y.head()

0    72
1    69
2    90
3    47
4    76
Name: math_score, dtype: int64

In [16]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch type                   1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   reading_score                1000 non-null   int64
 6   writing_score                1000 non-null   int64
dtypes: int64(2), str(5)
memory usage: 54.8 KB


In [18]:
categorical_cols = X.select_dtypes(include=['str']).columns
for col in categorical_cols:
    print(f"unique values in {col} : {X[col].unique()}")

unique values in gender : <StringArray>
['female', 'male']
Length: 2, dtype: str
unique values in race/ethnicity : <StringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
unique values in parental level of education : <StringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str
unique values in lunch type : <StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
unique values in test preparation course : <StringArray>
['none', 'completed']
Length: 2, dtype: str


In [19]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [24]:
# Create column transformer with 3 types of transformers

num_feature = X.select_dtypes(exclude=['str']).columns
cat_feature = X.select_dtypes(include=['str']).columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    [
     ("OneHotEncoder", categorical_transformer, cat_feature),
     ("StandardScaler", numeric_transformer, num_feature)
    ]
)

In [25]:
X = preprocessor.fit_transform(X)

In [30]:
X

array([[ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.19399858,  0.39149181],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.42747598,  1.31326868],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.77010859,  1.64247471],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.12547206, -0.20107904],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.60515772,  0.58901542],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.15336989,  1.18158627]], shape=(1000, 19))

In [31]:
# Create a train-test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 19), (200, 19), (800,), (200,))

In [29]:
X.shape

(1000, 19)